# Machine Learning Project
# Getting Card Information
# Kylle Waldie

# Pokemon Grading Tool

## Ebay API

### Imports

In [1]:
import requests
import base64
import csv
import re
import os
from dotenv import load_dotenv
import numpy as np

### API

In [2]:
load_dotenv()  # loads .env into environment variables

CLIENT_ID = os.getenv("EBAY_CLIENT_ID")
CLIENT_SECRET = os.getenv("EBAY_CLIENT_SECRET")
MARKETPLACE_ID = "EBAY_US"

print("Client ID loaded:", CLIENT_ID is not None)
print("Client Secret loaded:", CLIENT_SECRET is not None)


Client ID loaded: True
Client Secret loaded: True


### Setting up

In [3]:
BASE_DIR       = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\raw"
CSV_PATH       = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\psa_grades_images.csv"
TOTAL_TO_FETCH = 100   # Total listings to fetch per grade (max eBay allows is 10,000)
LIMIT          = 100    # Items per API request (eBay max is 100)
VALID_GRADES   = {str(i) for i in range(1, 11)}

### Authorization

In [4]:
def get_ebay_access_token(client_id, client_secret):
    credentials = f"{client_id}:{client_secret}"
    encoded = base64.b64encode(credentials.encode()).decode()

    url = "https://api.ebay.com/identity/v1/oauth2/token"

    headers = {
        "Authorization": f"Basic {encoded}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    data = {
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope"
    }

    response = requests.post(url, headers=headers, data=data)
    response.raise_for_status()

    return response.json()["access_token"]

### eBay Search

In [5]:
def search_psa_items(access_token, query="Pokemon PSA graded", limit=100, offset=0, marketplace_id="EBAY_US"):
    import requests

    url = "https://api.ebay.com/buy/browse/v1/item_summary/search"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "X-EBAY-C-MARKETPLACE-ID": marketplace_id
    }

    params = {
        "q": query,
        "limit": limit,
        "offset": offset
    }

    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json().get("itemSummaries", [])


### Grade Extraction

In [6]:
def extract_psa_grade(title):
    match = re.search(r'PSA\s?-?\s?(\d{1,2})', title, re.IGNORECASE)
    if match:
        grade = match.group(1)
        return grade if grade in VALID_GRADES else None
    return None

### Higher Res Images

In [7]:
def get_highres_image_url(url):
    # Replace small-size suffix with s-l1600 for best standard resolution
    return re.sub(r's-l\d+\.', 's-l1600.', url)

### Extract Grade & Images

In [8]:
def extract_grade_and_images(items):
    results = []

    for item in items:
        title = item.get("title", "")
        grade = extract_psa_grade(title)

        if not grade:
            continue

        image_url = item.get("image", {}).get("imageUrl")

        if image_url:
            image_url = get_highres_image_url(image_url)  # always grab high res
            results.append((grade, image_url))
    return results

### Saving CSV File

In [9]:
def save_to_csv(data, filename="psa_grades_images.csv"):
    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        # Write header only if file does not exist yet
        if not file_exists:
            writer.writerow(["PSA_Grade", "Image_URL"])

        writer.writerows(data)

### Download Images

In [10]:
def download_image(url, filepath):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        with open(filepath, "wb") as f:
            f.write(response.content)

        return True
    except Exception as e:
        print(f"Failed to download {url}: {e}")
        return False

### Download By PSA

In [11]:
def download_psa_images(psa_data, base_dir="dataset/raw"):
    os.makedirs(base_dir, exist_ok=True)
    
    # Start each grade counter based on files already in that folder
    counters = {}
    for grade, _ in psa_data:
        if grade not in counters:
            grade_dir = os.path.join(base_dir, f"PSA_{grade}")
            os.makedirs(grade_dir, exist_ok=True)
            existing = [f for f in os.listdir(grade_dir) if f.lower().endswith('.jpg')]
            counters[grade] = len(existing)  # start counting from what's already there
    downloaded = {grade: 0 for grade in counters}
    failed = {grade: 0 for grade in counters}

    for grade, image_url in psa_data:
        grade_dir = os.path.join(base_dir, f"PSA_{grade}")
        counters[grade] += 1
        filename = f"psa_{grade}_{counters[grade]}.jpg"
        filepath = os.path.join(grade_dir, filename)
        
        if os.path.exists(filepath):
            continue
        success = download_image(image_url, filepath)
        if success:
            downloaded[grade] += 1
        else:
            failed[grade] += 1

    return downloaded, failed

### Filter by grade

In [12]:
def filter_by_grade(psa_data, target_grade):
    """Return only items with the specified PSA grade."""
    return [item for item in psa_data if item[0] == str(target_grade)]


### Running Funcs

In [15]:
print("\nAuthenticating with eBay API...")
access_token = get_ebay_access_token(CLIENT_ID, CLIENT_SECRET)
print("Authentication successful!\n")

total_downloaded = 0
total_failed     = 0

# Loop through all PSA grades 1-10
for grade in range(1, 11):
    print(f"{'='*45}")
    print(f"Fetching PSA {grade} listings...")

    # Paginate through eBay results
    all_items = []
    for offset in range(0, TOTAL_TO_FETCH, LIMIT):
        items = search_psa_items(
            access_token,
            query=f"Pokemon PSA grade {grade}",
            limit=LIMIT,
            offset=offset
        )
        all_items.extend(items)
        if len(items) < LIMIT:
            break  # No more results available

    # Extract grades and image URLs
    psa_data = extract_grade_and_images(all_items)

    # Filter to only keep items that actually match this grade
    psa_data = [(g, url) for g, url in psa_data if g == str(grade)]

    print(f"Found {len(psa_data)} PSA {grade} images from {len(all_items)} listings.")

    # Save URLs to CSV
    save_to_csv(psa_data)

    # Download images
    downloaded, failed = download_psa_images(psa_data)
    grade_downloaded = downloaded.get(str(grade), 0)
    grade_failed = failed.get(str(grade), 0)

    print(f"Downloaded: {grade_downloaded}")
    if grade_failed > 0:
        print(f"Failed:{grade_failed}")

    total_downloaded += grade_downloaded
    total_failed += grade_failed

# Final summary
print(f"\n{'='*45}")
print(f"All grades complete!")
print(f"Total downloaded: {total_downloaded}")
if total_failed > 0:
    print(f"Total failed: {total_failed}")
print(f"Images saved to: {BASE_DIR}")
print(f"URLs saved to: {CSV_PATH}")
print(f"{'='*45}")


Authenticating with eBay API...
Authentication successful!

Fetching PSA 1 listings...
Found 87 PSA 1 images from 5000 listings.
Failed to download https://i.ebayimg.com/images/g/ResAAeSwEP1o6Lz1/s-l1600.jpg: HTTPSConnectionPool(host='i.ebayimg.com', port=443): Read timed out.
Downloaded: 86
Failed:1
Fetching PSA 2 listings...
Found 35 PSA 2 images from 5000 listings.
Downloaded: 35
Fetching PSA 3 listings...
Found 708 PSA 3 images from 5000 listings.
Failed to download https://i.ebayimg.com/images/g/RXQAAeSwqnBo0Mxm/s-l1600.jpg: HTTPSConnectionPool(host='i.ebayimg.com', port=443): Read timed out.
Failed to download https://i.ebayimg.com/images/g/n-sAAeSwZpxpZCoP/s-l1600.jpg: HTTPSConnectionPool(host='i.ebayimg.com', port=443): Read timed out.
Failed to download https://i.ebayimg.com/images/g/Ws0AAeSwKVBpfDKG/s-l1600.jpg: HTTPSConnectionPool(host='i.ebayimg.com', port=443): Read timed out.
Downloaded: 705
Failed:3
Fetching PSA 4 listings...
Found 651 PSA 4 images from 5000 listings.
F

HTTPError: 401 Client Error: Unauthorized for url: https://api.ebay.com/buy/browse/v1/item_summary/search?q=Pokemon+PSA+grade+10&limit=100&offset=0